In [12]:
#Первичный анализ конкурентов (буду дорабатывать)
#Пока что тут только конкуренты в сфере образования. Далее сделаю отдельный анализ и по конкурентам в сфере визовых услуг
!pip -q install pandas beautifulsoup4 requests lxml plotly

import re
import time
import uuid
import requests
import pandas as pd
import plotly.express as px

from bs4 import BeautifulSoup
from urllib.parse import urljoin

#вручную выписываем конкурентов
competitors = [
    {
        "name": "UniPage",
        "url": "https://www.unipage.net/ru",
        "type": "Образовательное агентство",
        "description": "Помощь с поступлением в зарубежные университеты."
    },
    {
        "name": "STAR Academy",
        "url": "https://www.staracademy.ru/",
        "type": "Образовательное агентство",
        "description": "Подбор программ обучения за рубежом и сопровождение поступления."
    },
    {
        "name": "Allterra Education",
        "url": "https://allterra.ru/",
        "type": "Образовательное агентство",
        "description": "Консультации и сопровождение поступления за рубеж."
    },
    {
        "name": "IQ Consultancy",
        "url": "https://iqconsultancy.ru/",
        "type": "Образовательное агентство",
        "description": "Поступление в зарубежные школы и университеты."
    }
]

# По этим словам определяем услуги
service_keywords = {
    "Визы": ["виза", "визу", "визовый", "шенген", "консульство", "посольство"],
    "Подача документов": ["подача документов", "подать документы", "пакет документов", "сбор документов"],
    "Поступление в вуз": ["поступление", "университет", "вуз", "бакалавриат", "магистратура"],
    "Консультации": ["консультация", "консультации", "консультант"],
    "Переводы/апостиль": ["перевод документов", "апостиль", "легализация", "нотариальный перевод"]
}

# По этим словам определяем страны и направления
country_keywords = {
    "США": ["сша", "usa", "america", "америка"],
    "Канада": ["канада", "canada"],
    "Великобритания": ["великобритания", "англия", "uk", "united kingdom"],
    "Германия": ["германия", "germany"],
    "Франция": ["франция", "france"],
    "Италия": ["италия", "italy"],
    "Испания": ["испания", "spain"],
    "Китай": ["китай", "china"],
    "Шенген": ["шенген", "schengen"]
}

# По этим словам определяем действия
actions_keywords = {
    "Онлайн-заявка": ["оставить заявку", "онлайн-заявка", "заявка онлайн", "подать заявку"],
    "Калькулятор": ["калькулятор", "рассчитать стоимость", "расчет стоимости"],
    "Личный кабинет": ["личный кабинет", "войти", "login"],
    "Чеклист": ["чеклист", "список документов", "перечень документов"]
}

#пишем вспомогательные классы

#очищаем текст от пробелов
def clean_text(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip()

#ищем какие группы ключевых слов встретились в тексте
def find_matches(text, keyword_dict):
    text = text.lower()
    found = []

    for group_name, words in keyword_dict.items():
        if any(word.lower() in text for word in words):
            found.append(group_name)

    return found

#ищем что-то похожее на цены
def find_prices(text):
    patterns = [
        r"\d[\d\s]{2,}\s?₽",
        r"\d[\d\s]{2,}\s?руб",
        r"от\s?\d[\d\s]{2,}",
        r"\d[\d\s]{1,}\s?€",
        r"\$\s?\d[\d\s]{1,}"
    ]

    prices = []

    for pattern in patterns:
        prices.extend(re.findall(pattern, text.lower()))

    prices = [clean_text(p) for p in prices]

    return list(set(prices))[:10]

#пишем класс, который занимается анализом и выставляет оценку каждому конкуренту
class SimpleCompetitorAnalyzer:

    def __init__(self, competitors):
        self.competitors = competitors

        self.headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 Chrome/120 Safari/537.36"
            ),
            "Accept-Language": "ru-RU,ru;q=0.9,en;q=0.8"
        }

    #загружаем страницу
    def load_page_text(self, url):
        try:
            response = requests.get(
                url,
                headers=self.headers,
                timeout=15,
                allow_redirects=True
            )

            if response.status_code >= 400:
                return "", f"HTTP {response.status_code}"

            response.encoding = response.apparent_encoding
            soup = BeautifulSoup(response.text, "lxml")

            # удаляем технические теги
            for tag in soup(["script", "style", "noscript"]):
                tag.extract()

            title = soup.title.text if soup.title else ""
            text = clean_text(soup.get_text(" "))
            return text, ""

        except Exception as e:
            return "", str(e)

    #анализ отдельного конкурента
    def analyze_one(self, competitor):
        text, error = self.load_page_text(competitor["url"])

        services = find_matches(text, service_keywords)
        countries = find_matches(text, country_keywords)
        actions = find_matches(text, actions_keywords)
        prices = find_prices(text)

        has_prices = len(prices) > 0
        has_free_consultation = "бесплатная консультация" in text.lower()


        #выставляем приблизительную оценку
        score = 0
        score += 10 if text else 0
        score += len(services) * 5
        score += len(countries) * 2
        score += 10 if has_prices else 0
        score += 10 if has_free_consultation else 0
        score += len(actions) * 7

        return {
            "id": str(uuid.uuid4()),
            "name": competitor["name"],
            "url": competitor["url"],
            "type": competitor["type"],
            "description": competitor["description"],
            "status": "ok" if text else "error",
            "error": error,
            "services_found": services,
            "countries_found": countries,
            "actions_found": actions,
            "prices_found": prices,
            "has_prices": has_prices,
            "has_free_consultation": has_free_consultation,
            "services_count": len(services),
            "countries_count": len(countries),
            "actions_count": len(actions),
            "score": score,
            "text_sample": text[:500]
        }
    #запускаем анализ всех конкурентов
    def run(self):
        results = []

        for i, competitor in enumerate(self.competitors, start=1):
            print(f"[{i}/{len(self.competitors)}] Анализируем: {competitor['name']}")

            result = self.analyze_one(competitor)
            results.append(result)

            print(f"  Статус: {result['status']}")
            print(f"  Услуги: {result['services_found']}")
            print(f"  Страны: {result['countries_found']}")
            print(f"  Действия: {result['actions_found']}")
            print()
            time.sleep(1)

        return pd.DataFrame(results)

#анализ
analyzer = SimpleCompetitorAnalyzer(competitors)
df = analyzer.run()

# Основная таблица для просмотра.
main_columns = [
    "name",
    "type",
    "status",
    "services_found",
    "countries_found",
    "actions_found",
    "has_prices",
    "has_free_consultation",
    "score",
    "error"
]

df_main = df[main_columns].sort_values("score", ascending=False)
df_main


[1/4] Анализируем: UniPage
  Статус: ok
  Услуги: ['Визы', 'Подача документов', 'Поступление в вуз', 'Консультации']
  Страны: ['США', 'Канада', 'Великобритания', 'Германия', 'Франция', 'Италия', 'Испания', 'Китай', 'Шенген']
  Действия: ['Онлайн-заявка', 'Личный кабинет']

[2/4] Анализируем: STAR Academy
  Статус: ok
  Услуги: ['Поступление в вуз']
  Страны: ['США', 'Канада', 'Великобритания', 'Германия', 'Франция', 'Италия', 'Испания', 'Китай']
  Действия: ['Онлайн-заявка']

[3/4] Анализируем: Allterra Education
  Статус: ok
  Услуги: []
  Страны: ['Великобритания']
  Действия: []

[4/4] Анализируем: IQ Consultancy
  Статус: ok
  Услуги: ['Визы', 'Поступление в вуз', 'Консультации']
  Страны: ['США', 'Канада', 'Великобритания', 'Германия', 'Франция', 'Италия', 'Испания']
  Действия: ['Онлайн-заявка']



,name,type,status,services_found,countries_found,actions_found,has_prices,has_free_consultation,score,error
0,UniPage,Образовательное агентство,ok,"[Визы, Подача документов, Поступление в вуз, К...","[США, Канада, Великобритания, Германия, Франци...","[Онлайн-заявка, Личный кабинет]",True,True,82,
3,IQ Consultancy,Образовательное агентство,ok,"[Визы, Поступление в вуз, Консультации]","[США, Канада, Великобритания, Германия, Франци...",[Онлайн-заявка],False,False,46,
1,STAR Academy,Образовательное агентство,ok,[Поступление в вуз],"[США, Канада, Великобритания, Германия, Франци...",[Онлайн-заявка],False,False,38,
2,Allterra Education,Образовательное агентство,ok,[],[Великобритания],[],False,False,12,


In [13]:
#пытаемся построить портреты конкурентов
def make_portrait(row):
    strengths = []
    weaknesses = []

    if row["has_prices"]:
        strengths.append("есть цены/что-то похожее на цены")
    else:
        weaknesses.append("нет цен")

    if row["has_free_consultation"]:
        strengths.append("есть бесплатная консультация")

    if row["actions_count"] > 0:
        strengths.append("есть онлайн действия")
    else:
        weaknesses.append("нет онлайн действий")

    if row["services_count"] >= 3:
        strengths.append("широкий набор услуг")

    if row["countries_count"] >= 4:
        strengths.append("широкий набор стран")

    #где мы можем их превосходить
    opportunity = []

    if not row["has_prices"]:
        opportunity.append("показать прозрачные тарифы")

    if row["actions_count"] == 0:
        opportunity.append("сделать чеклист/калькулятор/личный кабинет")

    if row["services_count"] < 3:
        opportunity.append("лучше раскрыть процесс подачи документов")

    if not opportunity:
        opportunity.append("конкурировать через узкую специализацию и скорость")

    return {
        "name": row["name"],
        "type": row["type"],
        "description": row["description"],
        "strengths": ", ".join(strengths) if strengths else "не получилось найти кодом",
        "weaknesses": ", ".join(weaknesses) if weaknesses else "не получилось найти кодом",
        "opportunity_for_us": ", ".join(opportunity),
        "short_portrait": (
            f"{row['name']} — {row['type']}. "
            f"Найденные услуги: {', '.join(row['services_found']) if row['services_found'] else 'не определены'}. "
            f"Найденные страны: {', '.join(row['countries_found']) if row['countries_found'] else 'не определены'}. "
            f"Возможность для нас: {', '.join(opportunity)}."
        )
    }


portraits_df = pd.DataFrame([make_portrait(row) for _, row in df.iterrows()])
portraits_df

#Хотим попробовать проверить несколько гипотез
total = len(df)
hypotheses = pd.DataFrame([
    {
        "hypothesis": "У большинства конкурентов сложно найти цены (частая проблема, которая всех бесит)",
        "metric": "Доля сайтов без цен",
        "value": round(1 - df["has_prices"].mean(), 2),
        "quick_interpretation": "Если значение высокое, можно выделиться за счет наличия цен в открытом доступе"
    },
    {
        "hypothesis": "У конкурентов мало онлайн продуктов (пример онлайн продукта: рассчитать стоимость. Это что-то что пользователь сам делает на сайте)",
        "metric": "Доля сайтов без найденных целевых действий",
        "value": round((df["actions_count"] == 0).mean(), 2),
        "quick_interpretation": "Если значение высокое, можно выделиться широким спектром услуг (универсальностью)"
    },
    {
        "hypothesis": "Бесплатная консультация часто используется как инструмент привлечения / удержания клиента",
        "metric": "Доля сайтов с бесплатной консультацией",
        "value": round(df["has_free_consultation"].mean(), 2),
        "quick_interpretation": "Если значение высокое, бесплатной консультации недостаточно (она есть у всех)"
    }
])

hypotheses

,hypothesis,metric,value,quick_interpretation
0,У большинства конкурентов сложно найти цены (ч...,Доля сайтов без цен,0.75,"Если значение высокое, можно выделиться за сче..."
1,У конкурентов мало онлайн продуктов (пример он...,Доля сайтов без найденных целевых действий,0.25,"Если значение высокое, можно выделиться широки..."
2,Бесплатная консультация часто используется как...,Доля сайтов с бесплатной консультацией,0.25,"Если значение высокое, бесплатной консультации..."


In [14]:
# 1.типы конкурентов
fig = px.histogram(
    df,
    x="type",
    title="Типы конкурентов в выборке",
    labels={"type": "Тип конкурента"}
)
fig.show()


# 2. сравнение конкурентов по набранным баллам
fig = px.bar(
    df.sort_values("score", ascending=True),
    x="score",
    y="name",
    color="type",
    orientation="h",
    title="Поверхностный score конкурентов",
    labels={
        "score": "Score",
        "name": "Конкурент",
        "type": "Тип"
    }
)
fig.show()


# 3. наличие цен, консультаций и действий
features_summary = pd.DataFrame({
    "feature": [
        "Есть цены",
        "Есть бесплатная консультация",
        "Есть "
    ],
    "share": [
        df["has_prices"].mean(),
        df["has_free_consultation"].mean(),
        (df["actions_count"] > 0).mean()
    ]
})

fig = px.bar(
    features_summary,
    x="feature",
    y="share",
    title="Доля конкурентов с ключевыми признаками",
    labels={
        "feature": "Признак",
        "share": "Доля конкурентов"
    },
    text="share"
)
fig.show()

features_summary


# 4. сколько услуг у каждого конкурента
fig = px.bar(
    df.sort_values("services_count", ascending=True),
    x="services_count",
    y="name",
    color="type",
    orientation="h",
    title="Количество типов услуг у конкурентов",
    labels={
        "services_count": "Количество типов услуг",
        "name": "Конкурент"
    }
)
fig.show()